In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark", "datasets", "wordcloud", "mlflow"])
    from google.colab import drive
    drive.mount("/content/drive")

import os
if IN_COLAB:
    os.chdir("/content/drive/MyDrive/amazon-reviews-sentiment-analysis")
else:
    # Ensure src/ is importable
    sys.path.insert(0, os.path.abspath("..")) if os.path.basename(os.getcwd()) == "notebooks" else None

import pandas as pd
import matplotlib.pyplot as plt

# Error Analysis

In [ ]:
from src.config import ProjectConfig
from src.data_loader import get_spark_session, load_data
from pyspark.ml import PipelineModel

config = ProjectConfig()
spark = get_spark_session(config.spark)
train_df, test_df = load_data(spark, config.data)

model = PipelineModel.load("app/model_artifacts/spark_model")
predictions = model.transform(test_df)
predictions.cache()

## Misclassification Overview

In [ ]:
total = predictions.count()
correct = predictions.filter("label = prediction").count()
incorrect = total - correct
print(f"Total: {total:,}")
print(f"Correct: {correct:,} ({correct/total*100:.2f}%)")
print(f"Incorrect: {incorrect:,} ({incorrect/total*100:.2f}%)")

## False Positives (Predicted Positive, Actually Negative)

In [ ]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import FloatType

extract_prob = udf(lambda v: float(v[1]), FloatType())

fp = predictions.filter("label = 0 AND prediction = 1") \
    .withColumn("confidence", extract_prob("probability")) \
    .select("combined_text", "confidence") \
    .orderBy(col("confidence").desc()) \
    .limit(20).toPandas()

for i, row in fp.iterrows():
    print(f"\n--- FP #{i+1} (confidence: {row['confidence']:.4f}) ---")
    print(row["combined_text"][:300])

## False Negatives (Predicted Negative, Actually Positive)

In [ ]:
fn = predictions.filter("label = 1 AND prediction = 0") \
    .withColumn("confidence", extract_prob("probability")) \
    .select("combined_text", "confidence") \
    .orderBy(col("confidence").asc()) \
    .limit(20).toPandas()

for i, row in fn.iterrows():
    print(f"\n--- FN #{i+1} (confidence: {row['confidence']:.4f}) ---")
    print(row["combined_text"][:300])

## Confidence Distribution

In [ ]:
import matplotlib.pyplot as plt

pdf = predictions.withColumn("confidence", extract_prob("probability")) \
    .withColumn("is_correct", (col("label") == col("prediction")).cast("int")) \
    .select("confidence", "is_correct").toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(pdf[pdf["is_correct"]==1]["confidence"], bins=50, alpha=0.6, label="Correct", color="green")
ax.hist(pdf[pdf["is_correct"]==0]["confidence"], bins=50, alpha=0.6, label="Incorrect", color="red")
ax.set_xlabel("Predicted Probability (Positive Class)")
ax.set_ylabel("Count")
ax.set_title("Confidence Distribution: Correct vs Incorrect Predictions")
ax.legend()
plt.tight_layout()
plt.savefig("docs/confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Accuracy by Review Length

In [ ]:
from pyspark.sql.functions import length, when

length_df = predictions.withColumn("text_length", length("combined_text")) \
    .withColumn("length_bucket",
        when(col("text_length") < 100, "0-100")
        .when(col("text_length") < 300, "100-300")
        .when(col("text_length") < 500, "300-500")
        .when(col("text_length") < 1000, "500-1000")
        .otherwise("1000+")
    ) \
    .withColumn("is_correct", (col("label") == col("prediction")).cast("int"))

bucket_acc = length_df.groupBy("length_bucket").agg(
    {"is_correct": "avg"}
).toPandas()
bucket_acc.columns = ["length_bucket", "accuracy"]

order = ["0-100", "100-300", "300-500", "500-1000", "1000+"]
bucket_acc["length_bucket"] = pd.Categorical(bucket_acc["length_bucket"], categories=order, ordered=True)
bucket_acc = bucket_acc.sort_values("length_bucket")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(bucket_acc["length_bucket"], bucket_acc["accuracy"], marker="o", lw=2, color="steelblue")
ax.set_xlabel("Review Length (characters)")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy by Review Length")
ax.set_ylim(0.8, 1.0)
plt.tight_layout()
plt.savefig("docs/accuracy_by_length.png", dpi=150, bbox_inches="tight")
plt.show()

## Failure Pattern Summary

Common failure patterns observed:
- **Sarcasm and irony:** Model struggles with sarcastic reviews where surface-level words are positive but meaning is negative
- **Mixed sentiment:** Reviews that discuss both pros and cons
- **Short reviews:** Very short reviews lack enough context for confident classification
- **Domain-specific language:** Technical or niche product terminology

In [ ]:
spark.stop()